# 1. 환경 준비

1. **`import matplotlib.pyplot as plt`**
    - 파이썬 시각화 라이브러리 
2. **`import seaborn as sns`**
    - `seaborn`은 `matplotlib`을 기반으로 만들어진 라이브러리
    - matplotlib 보다 복잡한 통계 그래프를 더욱 쉽게 그려줌
    - `set_theme()`: `seaborn`이 가진 기본 테마를 적용 (matplotlib)으로 그린 그래프에 스타일 부여

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

import matplotlib.pyplot as plt
import seaborn as sns

# seaborn의 기본 테마 설정
sns.set_theme()

# 데이터 불러오기
# as_frame=True: DataFrame 형태로 반환
# return_X_y=True: 특성 행렬과 타겟 벡터를 별도로 반환
df, y = load_wine(as_frame=True, return_X_y=True)

# 타겟 열 추가
df["quality"] = y

# 2. **sns.heatmap 그려보기**

1. 모든 변수들 사이 상관관계를 `corr` 에 담기
    - `df.corr()`: 모든 특성간의 상관관계
2. seaborn을 사용해서, sns.heatmap을 그려보기
    - `sns.heatmap(dataframe)`
- heatmap이란?
    - **데이터의 패턴을 한눈에 파악하기 위해** 만들어진 시각화 도구
    - 기상청의 날씨 지도를 예로, 온도가 **높은 지역은 붉은색**으로, **온도가 낮은 지역은 푸른색**으로 표시
    - **상관관계가 강할수록 짙은 색, 약할수록 옅은 색**

In [ ]:
# 1. 모든 변수들 사이 상관관계를 `corr` 에 담기
corr = df.corr()

# 2. seaborn을 사용해서, sns.heatmap을 그려보기
sns.heatmap(corr)

# 3. 더 명확하게 표기하기

1. Figure size를 키우기
    - `plt.figure(가로, 세로)`: 크기 단위는 인치
2. 상관 관계가 몇 점인지 그래프에 표기하기
    - `annot=True`: 히트맵의 각 셀 안에 숫자를 표시 (주석)
3. 불필요한 데이터 지우기
    - `mask={df}`: 인자로 받은 데이터를 **마스킹(가려서)**하여 표기
    1. 자기 자신과의 상관 관계와 우상단의 정보 그래프에서 지우기
        - 자기 자신 (0, 0), (1, 1)은 자기 자신이므로 항상 1로 고정. **`불필요한 데이터`**
        - 상관관계 행렬은 대각선을 기준으로 윗쪽과 아랫쪽이 동일함. **`불필요한 데이터`**
    2. 그래프에서 지울 대상을 먼저 정해야 함.
        - 상관관계 행렬과 동일한 크기의 마스킹용 행렬을 만들기
        1. `np.ones_like()`
        2. `np.triu()` 행렬의 위쪽 삼각형만 남기고, 나머지를 모두 0으로 변환
        - 위에서 만든 `ones_like` df를 가지고 `triu`를 사용한 결과는?
            - 자기자신을 기준으로 위쪽은 모두 1, 나머지는 모두 0
    - **마스킹이란?** 특정 조건에 맞는 데이터만 선택적으로 드러내거나 가리기
        - 위에서 만들어낸 1로만 가득 찬 윗부분 삼각형을 마스크로 사용해서 
        `sns.heatmap(mask=DataFrame)`: 으로 넘겨준 데이터의 True인 부분을 마스킹 하겠다.
4. 상관 계수 값을 소숫점 2자리까지만 표기
    - `fmt='.2f'`: 그 값을 소숫점 2자리 까지만 보여 달라.
5. 색 구분이 조금 더 명확해 지게 변경
    - 참고: https://seaborn.pydata.org/tutorial/color_palettes.html
    - `cmap='coolwarm'` : 색상 팔레트를 지정.
        - **양수면 붉은색, 음수면 푸른색**
        - 오해 금지, 상관관계가 높으면 붉은색이 아니라, 
        양수 값이 1에 가까워 질 수록 붉어지고,
        음수 값이 -1에 가까워 질 수록 푸른색이 되는 것

In [ ]:
# Figure size를 키웁니다.
plt.figure(figsize=(10, 8))

# 상관 관계 행렬과 동일한 크기의 1로 가득찬 마스킹용 행렬 만들기
mask_mat = np.ones_like(corr, dtype=int)

# 위쪽 삼각형 마스크를 만듭니다.
mask = np.triu(mask_mat)
print(mask)

# 히트맵을 그립니다.
sns.heatmap(
    corr, 
    annot=True, 
    fmt='.2f', 
    cmap='coolwarm', 
    mask=mask
)
plt.grid(False)
plt.show()

# 4. 실습: 분포 살펴보기
1. **데이터가 주로 어디에 모여있는가?** (가장 흔한 값은 무엇인가)
2. **데이터가 얼마나 넓게 퍼져 있는가?** (데이터의 범위는 어느 정도인가)
3. **데이터에 특이한 값(이상치)이 있는가?** (평균에서 아주 멀리 떨어진 값이 있는가)
    - 이 과정을 통해 모델을 만들 때, **어떤 변수가 중요한지, 어떤 변수를 수정해야 할지** 짐작

- **`sns.histplot(data=df, x='flavanoids', bins=20, kde=True)`**
- `x='flavanoids'` : ‘flavanoids’ 열의 분포를 그려라. (x 값이 열, column)
- `bins=20`: 20개의 구간
- `kde=True` (Kernel Density Estimate): 히스토그램 막대 그래프위로 부드러운 곡선 그리기

In [ ]:
# figsize 조절 (a=가로, b=세로) 단위: inch
plt.figure(figsize=(6, 4))
# 히스토그램 그리기 (x=컬럼명, bins=막대 개수, kde=커널 밀도 추정선)
    # 히스토그램은 bins의 개수와 위치에 따라서 모양이 극단적으로 표기될 수도 있음.
    # 커널 밀도 추정선은 부드러운 곡선을 통해 데이터의 분포를 보다 직관적으로 이해할 수 있게 도와줌.
sns.histplot(data=df, x='flavanoids', bins=20, kde=True)
plt.title('Flavanoids Distribution')
plt.show()

## 4-1 여러 그래프 그리기
- 아래 각 과정을 ax (개별 그래프 0, 1에 삽입)
1. 클래스 별 flavanoids 분포도
    - `hue`: 색조
        - 이곳에 기준이 될 값을 삽입. (여기서는 클래스 즉, quality)
2. 히스토그램에 다양한 정보 표기하기
    - 각각, x와 y에 값 삽입 후, hue 설정

In [ ]:
# fig, ax = plt.subplots(figsize=(18, 5), ncols=2):
    # plt.subplots(ncols=3): 여러 그래프를 한 그림에 동시에. ncols 개수 만큼 나타낼 것
    # fig: 그래프 전체
    # ax: 개별적인 그래프들 (0, 1) 번째에 표기할 것
fig, ax = plt.subplots(figsize=(10, 5), ncols=2)

# 1. 클래스 별 flavanoids 분포도
sns.histplot(data=df, x='flavanoids', bins=20, kde=True, hue='quality', ax=ax[0])

# 2. 히스토그램에 다양한 정보 표기하기
    # x='flavanoids', y='total_phenols': 각각 x와 y에 값 삽입
    # hue='quality': 클래스 별로 색상 구분
sns.histplot(data=df, x='flavanoids', y='total_phenols', bins=20, hue='quality', ax=ax[1])

# fig.tight_layout(): 서로 겹치지 않게 자동으로 간격 조절
fig.tight_layout()

## 4-2. 산점도 그리기
- 산점도(scatterplot)란? 
  - 두 변수 간의 관계를 점(dot)으로 표현
  - 히스토그램은 한 변수의 `분포`를 보여주는 그림이었다면, **산점도**는 두 변수고 서로에게 어떤 영향을 미치는지, `관계`를 보여주는 그림

In [ ]:
# 산점도 그리기
sns.scatterplot(data=df, x="flavanoids", y="total_phenols", hue="quality")
plt.show()

## 4-3. 특성별 산점도 그리기

- 모든 특성의 산점도를 하나씩 그리고, 비교하면 매우 번거로운 과정이 됨.
- 와인 등급과 상관 관계가 높은 5개의 데이터를 뽑아서, 각 특성들끼리의 산점도만 찍어보자.

1. 와인 등급과 상관 관계가 높은 5개의 특성 구하기
    - 주의: 상관 관계가 -1에 가까울 수록 음의 상관 관계가 높게 나타난다는 의미
    - 따라서 `절대값`으로 비교하여야 함.

In [ ]:
# 와인 등급에 영향을 미치는 특성 찾기
# 1. 와인 등급과의 상관관계 절대값 구하기
    # 예: -0.8은 0.5보다 더 강한 상관관계
abs_corr = df.corr()['quality'].abs()

# 2. 절대값이 높은 순서대로 내림 차순 정렬
top_features = abs_corr.sort_values(ascending=False)
print(top_features.head())

# 3. 상위 5개 (quality 제외)만 뽑기
top_features = top_features.index[1:6]

print(f'\n 상위 5개 특성: \n {top_features}')

1. `sns.pairplot(DataFrame)`
    - 여러 변수들 간의 관계를 한 번에 보여주는 그래프
    - pairplot은 여러 개의 산점도와 히스토그램이 한 표에 채워져서 그려짐
        - **대각선 그래프:** 각 변수의 분포(히스토그램)
        - **그 외:** 두 변수 간의 관계 (산점도)

In [ ]:
# hue='quality'를 사용해 와인 등급별로 색깔을 다르게 표시
# diag_kind: 대각선 그래프 종류 지정 (kde: 커널 밀도 추정선)
# corner=True로 대각선 아래쪽 그래프만 표시
sns.pairplot(
    data=df, 
    vars=top_features,
    hue='quality', 
    diag_kind='kde',
    corner=True
)

plt.show()

## 4-4. 결측치와 이상치 감지하기

- 무작위 위치에 결측치와 이상치를 생성할 것이지만, 
- **모두가 같은 결과를 볼 수 있도록 시드 고정!** `np.random.seed(42) `

1. 임의의 `결측치`와 `이상치`를 가지고 있는 데이터프레임 생성하기
  - 단, 원본을 훼손하지 않도록 복사본을 생성하여 진행
  - **결측치:** 빠진 데이터
    - 말 그대로 값이 없는 (누락된) 데이터 를 의미
  - **이상치:** 비정상적인 데이터

In [ ]:
# 모두가 같은 결과를 얻도록 시드 설정
np.random.seed(42)  

# 임의의 결측치와 이상치를 추가하기 위한 복사본 생성
df_missing = df.copy()

# 데이터프레임의 일부 인덱스를 무작위로 선택 하여 결측치와 이상치를 추가
    # size=10: 10개 선택
    # replace=False: 중복 없이 선택
missing_idx = np.random.choice(df_missing.index, size=10, replace=False)
print(f'결측치 인덱스: {missing_idx}')

# 결측치: flavanoids 컬럼의 일부 값을 NaN으로 변경
    # loc: 라벨 기반 인덱싱
df_missing.loc[missing_idx, 'flavanoids'] = np.nan

# 이상치: alcohol 컬럼의 일부 값을 비정상적으로 크게 설정
outlier_idx = np.random.choice(df_missing.index, size=5, replace=False)
print(f'이상치 인덱스: {outlier_idx}')

# 이상치를 평균의 5배로 설정
df_missing.loc[outlier_idx, 'alcohol'] = df_missing['alcohol'].mean() * 5

2. `결측치` 데이터 확인하는 두가지 방법
    1. 데이터프레임에서 합계 수치 확인하기
    2. seaborn으로 시각화 하여 확인하기

In [ ]:
# isnull(): 컬럼별 결측치 여부 확인
# sum(): 결측치 개수 합산
print(df_missing.isnull().sum())

# 시각화로 결측치 확인
# cbar=False: 컬러바 표시 안함
sns.heatmap(df_missing.isnull(), cbar=False)
plt.title("Visualize Missing Values")
plt.show()

### 4-4-1. 이상치 감지 기준 설정하기

1. **IQR (Interquartile Range)**
    - `사분위 범위`
    - **Q1 (1사분위수)**: 전체 데이터를 4등분했을 때, 25% 지점의 값.
    - **Q3 (3사분위수)**: 전체 데이터를 4등분했을 때, 75% 지점의 값.
    - **`IQR = Q3 - Q1`**: Q1과 Q3 사이의 거리를 계산
    
2. **이상치 감지 기준 (Outlier Detection)**
    - **하한선 (Lower Bound)**: `Q1 - 1.5 * IQR`
    - **상한선 (Upper Bound)**: `Q3 + 1.5 * IQR`

3. **박스플롯 (Boxplot)**
    - **박스 전체 영역**: Q1부터 Q3까지의 범위
    - **선 (수염)**: 상한선과 하한선까지의 범위
    - **점**: 이 상한선과 하한선을 벗어나는 **이상치**

In [ ]:
def detect_outliers_iqr(data, column):
    # IQR(Interquartile Range) 방법으로 이상치 감지
    # 1. Q1(25%)과 Q3(75%) 계산
        # quantile: 분위수 계산
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    # 2. IQR을 이용해 이상치 경계값 계산
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 3. 경계값을 벗어나는 데이터 필터링
        # data[column] < lower_bound: 경계값보다 작은 값
        # data[column] > upper_bound: 경계값보다 큰 값
        # |: or 연산자
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

# alcohol 컬럼에서 이상치 감지
outliers_alcohol = detect_outliers_iqr(df_missing, 'alcohol')
print(f"alcohol 이상치 개수: {len(outliers_alcohol)}")

# boxplot으로 이상치 확인
sns.boxplot(x=df_missing['alcohol'])
plt.title("Outliers of Alcohol")
plt.show()

# 5. 실습: 결측치와 이상치 처리하기
- **결측치 처리 방법**:
    - **삭제**
        - 데이터가 충분하면 결측치가 있는 행이나 열을 그냥 지워버리는 방법.
        - 쉽지만, 데이터가 많이 사라질 수 있으니 주의
    - **대체**
        - 비어있는 곳을 다른 값으로 채우는 방법.
            - 숫자 데이터: **평균이나 중앙값**으로 대체
            - 시계열 데이터: **앞뒤 값**으로 대체하거나 **머신러닝으로 예측**해서 대체
        
- **이상치 처리 방법**:
    - **삭제**: 명백한 오류라고 판단되면 그냥 삭제.
    - **변환**: 데이터의 모양을 바꿔서 이상치의 영향력을 줄이는 방법
    - **대체**: 이상치를 **특정 경계 값이나 평균, 중앙값** 같은 걸로 대체

In [ ]:
# wine 데이터는 결측치가 없어서, 아래 코드의 결과는 변함 없음.
# 결측치 처리 방법 1: 평균값으로 대체
df_filled = df_missing.fillna(df_missing.mean(numeric_only=True))
print('결측치 확인')
print(df_filled.isnull().sum())

# 이상치 처리 방법 1: IQR 기준 밖의 값 제거
    # ~: not 연산자
    # isin(): 특정 값이 포함되어 있는지 확인
df_no_outliers = df_filled[~df_filled.index.isin(outliers_alcohol.index)]

# 이상치: IQR 기준 밖의 값을 평균으로 대체
    # 1. alcohol 컬럼에서 이상치 감지
outliers_alcohol = detect_outliers_iqr(df_missing, 'alcohol')
    # 2. 이상치를 평균값으로 대체
mean_alcohol = df_filled['alcohol'].mean()
df_replaced_outliers = df_filled.copy()
# loc[행 인덱스, 열 이름]: 특정 위치의 값을 변경
df_replaced_outliers.loc[outliers_alcohol.index, 'alcohol'] = mean_alcohol
